# 03 内容融合实验

**目的**：四视图融合（含 ρ 自适应权重）、基于银标准的完整评测、预算实验、消融实验和可视化。

实现 CONTENT_VIEW_EXTENSION.md §5（融合）和 §6（实验）。

**对比方法**：
1. **Meta-only** — 现有三视图融合（`S_fused3_symrow`）
2. **Content-only** — 表格内容视图（`S_tabcontent_symrow`）
3. **Naive Fusion** — 0.5 * S_fused3 + 0.5 * S_tabcontent
4. **Adaptive Fusion** — ρ 自适应四视图权重（`S_fused4_symrow`）
5. **Adaptive+Consistency** — 含 c(i) 调节（`S_fused4c_symrow`）

**输出**：
- `tmp/content/S_naive4_symrow_k50_*`
- `tmp/content/S_fused4_symrow_k50_*`
- `tmp/content/S_fused4c_symrow_k50_*`
- `tmp/content/metrics_content_experiments.csv`
- `tmp/content/fig_*.png`

### 配置与导入

**功能**：导入所需库和自定义评测指标模块，定义路径常量、评测参数和统一权重

**输入**：
- `src/metrics.py` — 评测指标函数

**输出**：
- `N` / `K_EVAL` / `K_SIM` / `W_TAG` / `W_DESC` / `W_CRE` 等全局参数

In [1]:
# ---- 配置 ----
import os, sys, json, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy import sparse

# 路径配置（notebook 位于 notebooks/04_content/，项目根目录在两级之上）
ROOT        = Path(".").resolve().parent.parent

sys.path.insert(0, str(ROOT))
from src.metrics import (
    dcg_at_k, ndcg_at_k, average_precision_at_k, mrr_at_k,
    precision_at_k, recall_at_k
)
from src.content.similarity import (
    load_manifest_flexible, load_csr_from_manifest, save_partitioned_edges,
)
from src.content.fusion import (
    compute_rho, compute_adaptive_alpha, apply_consistency_adjustment, fuse_views,
)

warnings.filterwarnings("ignore", category=FutureWarning)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# 路径
TMP_DIR     = ROOT / "tmp"
CONTENT_DIR = TMP_DIR / "content"

# 参数
N       = 521735
K_EVAL  = 20
K_SIM   = 50
W_TAG   = 0.5
W_DESC  = 0.3
W_CRE   = 0.2
PART_SIZE = 2_000_000

print(f"N={N}, K_EVAL={K_EVAL}, K_SIM={K_SIM}")
print(f"Unified weights: W_TAG={W_TAG}, W_DESC={W_DESC}, W_CRE={W_CRE}")

N=521735, K_EVAL=20, K_SIM=50
Unified weights: W_TAG=0.5, W_DESC=0.3, W_CRE=0.2


### 通用工具函数

**功能**：定义三个核心工具函数——灵活加载 manifest、从 manifest 构建 top-K 邻居、从 manifest 加载 CSR 稀疏矩阵

**输入**：
- `prefix` / `base_dir` — 视图名称和基础目录

**输出**：
- `load_manifest_flexible` → `(manifest_dict, parts_list, parent_dir)`
- `build_topk_for_method` → `(nbr_idx, nbr_w)` 字典
- `load_csr_from_manifest` → scipy CSR 矩阵

In [2]:
# ---- 工具函数 ----
# load_manifest_flexible, load_csr_from_manifest, save_partitioned_edges
# are imported from src.content.similarity.

def build_topk_for_method(prefix, k_eval, base_dir):
    """
    加载相似度边并返回每个文档的 top-K 邻居。
    返回 (nbr_idx, nbr_w) 字典：{doc_idx: (邻居索引数组, 权重数组)}
    """
    man, parts, manifest_dir = load_manifest_flexible(prefix, base_dir, k=K_SIM)
    dfs = []
    for pf in parts:
        path = manifest_dir / pf
        if path.exists():
            dfs.append(pd.read_parquet(path, engine="fastparquet"))
    if not dfs:
        return {}, {}
    edges = pd.concat(dfs, ignore_index=True)

    # 按 row 分组，按 val 降序排列，取前 k_eval 个
    nbr_idx = {}
    nbr_w = {}
    for row_i, group in edges.groupby("row"):
        row_i = int(row_i)
        top = group.nlargest(k_eval, "val")
        nbr_idx[row_i] = top["col"].values.astype(np.int64)
        nbr_w[row_i] = top["val"].values.astype(np.float32)

    return nbr_idx, nbr_w

print("Utility functions defined.")

Utility functions defined.


### 分区边存储函数

**功能**：将 COO 稀疏矩阵按分区大小切分为多个 parquet 文件并生成 manifest.json

**输入**：
- `rows` / `cols` / `vals` — COO 格式的边数据
- `N` / `prefix` / `k` / `output_dir` — 矩阵参数和输出配置

**输出**：
- `{prefix}_k{k}_part*.parquet` — 分区边文件
- `{prefix}_k{k}_manifest.json` — 索引清单

In [3]:
# save_partitioned_edges is imported from src.content.similarity.
# See src/content/similarity.py for the implementation.
print("Using save_partitioned_edges from src.content.similarity")

Using save_partitioned_edges from src.content.similarity


### ρ 行信息浓度与自适应权重

**功能**：计算每个视图 CSR 矩阵的行信息浓度 ρ[i] = Σ_j S[i,j]²，然后按视图归一化为逐行自适应权重 α

**输入**：
- `S_csr` — scipy CSR 稀疏矩阵
- `rho_dict` — `{view_name: (N,) ρ 数组}` 字典

**输出**：
- `compute_rho` → (N,) ρ 数组
- `compute_adaptive_alpha` → `{view_name: (N,) α 权重数组}`

In [4]:
# compute_rho and compute_adaptive_alpha are imported from src.content.fusion.
# See src/content/fusion.py for the implementation.
print("Using compute_rho, compute_adaptive_alpha from src.content.fusion")

Using compute_rho, compute_adaptive_alpha from src.content.fusion


### 一致性调节

**功能**：根据一致性分数 c(i) 调节 tabcontent 视图的权重——低一致性文档降低内容视图贡献，调节后重新归一化所有视图权重

**输入**：
- `alpha` — `{view_name: (N,) α 数组}` 原始自适应权重
- `c_scores_df` — 一致性分数 DataFrame
- `N` — 节点总数
- `beta` — 调节强度参数（默认 0.5）

**输出**：
- `alpha_adj` — 调节并重新归一化后的权重字典

In [5]:
# apply_consistency_adjustment is imported from src.content.fusion.
# See src/content/fusion.py for the implementation.
print("Using apply_consistency_adjustment from src.content.fusion")

Using apply_consistency_adjustment from src.content.fusion


### 多视图融合函数

**功能**：逐行对多个视图矩阵进行加权求和，执行 top-K 截断和 L1 行归一化，内存友好的批处理实现

**输入**：
- `S_dict` — `{view_name: CSR 矩阵}` 视图矩阵字典
- `alpha_dict` — `{view_name: (N,) α 数组}` 权重字典
- `views` — 视图名称列表
- `N` / `K` / `batch_size` — 矩阵维度和处理参数

**输出**：
- 返回 COO 格式的 `(rows, cols, vals)` 三元组

In [6]:
# fuse_views is imported from src.content.fusion.
# See src/content/fusion.py for the implementation.
print("Using fuse_views from src.content.fusion")

Using fuse_views from src.content.fusion


### 加载四个视图矩阵 + 计算 ρ

**功能**：加载 tag/text/beh/tabcontent 四个视图的 CSR 矩阵以及三视图融合矩阵，计算每个视图的行信息浓度 ρ

**输入**：
- `tmp/S_tag_symrow_k50_*` — 标签视图
- `tmp/S_text_symrow_k50_*` — 文本视图
- `tmp/S_beh_symrow_k50_*` — 行为视图
- `tmp/content/S_tabcontent_symrow_k50_*` — 内容视图
- `tmp/S_fused3_symrow_k50_*` — 三视图融合

**输出**：
- `S_dict` — 四个视图的 CSR 矩阵字典
- `rho_dict` — 四个视图的 ρ 数组字典

In [7]:
# 加载四个 CSR 矩阵
print("Loading view matrices...")
S_tag = load_csr_from_manifest("S_tag_symrow", N, TMP_DIR)
print(f"  S_tag: nnz={S_tag.nnz:,}")

S_text = load_csr_from_manifest("S_text_symrow", N, TMP_DIR)
print(f"  S_text: nnz={S_text.nnz:,}")

S_beh = load_csr_from_manifest("S_beh_symrow", N, TMP_DIR)
print(f"  S_beh: nnz={S_beh.nnz:,}")

S_tabcontent = load_csr_from_manifest("S_tabcontent_symrow", N, CONTENT_DIR)
print(f"  S_tabcontent: nnz={S_tabcontent.nnz:,}")

S_fused3 = load_csr_from_manifest("S_fused3_symrow", N, TMP_DIR)
print(f"  S_fused3: nnz={S_fused3.nnz:,}")

S_dict = {
    "tag": S_tag,
    "text": S_text,
    "beh": S_beh,
    "tabcontent": S_tabcontent,
}
views_4 = ["tag", "text", "beh", "tabcontent"]

# 计算每个视图的 ρ
print("\nComputing rho...")
rho_dict = {}
for v in views_4:
    rho_dict[v] = compute_rho(S_dict[v])
    print(f"  rho_{v}: min={rho_dict[v].min():.6f}, "
          f"median={np.median(rho_dict[v]):.6f}, max={rho_dict[v].max():.6f}")

Loading view matrices...


  S_tag: nnz=28,159,756


  S_text: nnz=28,161,106


  S_beh: nnz=26,023,186
  S_tabcontent: nnz=76,454


  S_fused3: nnz=26,086,750

Computing rho...
  rho_tag: min=0.010916, median=0.018935, max=0.020222


  rho_text: min=0.010547, median=0.018935, max=0.020245
  rho_beh: min=0.000000, median=0.054806, max=1.000000
  rho_tabcontent: min=0.000000, median=0.000000, max=0.020208


### 朴素融合

**功能**：以等权 0.5 * S_fused3 + 0.5 * S_tabcontent 构建朴素融合矩阵

**输入**：
- `S_fused3` / `S_tabcontent` — 两个 CSR 矩阵

**输出**：
- `tmp/content/S_naive4_symrow_k50_*` — 朴素融合分区文件和 manifest

In [8]:
# ---- 构建融合变体 ----

# 1. 朴素融合：0.5 * S_fused3 + 0.5 * S_tabcontent
print("=" * 60)
print("Building Naive Fusion (0.5*S_fused3 + 0.5*S_tabcontent)...")
naive_alpha = {
    "fused3": np.full(N, 0.5, dtype=np.float64),
    "tabcontent": np.full(N, 0.5, dtype=np.float64),
}
naive_S_dict = {
    "fused3": S_fused3,
    "tabcontent": S_tabcontent,
}
naive_views = ["fused3", "tabcontent"]

naive_rows, naive_cols, naive_vals = fuse_views(
    naive_S_dict, naive_alpha, naive_views, N, K=K_SIM
)
save_partitioned_edges(
    naive_rows, naive_cols, naive_vals, N,
    prefix="S_naive4_symrow", k=K_SIM, output_dir=CONTENT_DIR,
    note="row-stochastic; 0.5*S_fused3 + 0.5*S_tabcontent; top-K + L1 norm"
)
del naive_rows, naive_cols, naive_vals

Building Naive Fusion (0.5*S_fused3 + 0.5*S_tabcontent)...


### 自适应融合

**功能**：使用 ρ 自适应权重对四个视图进行融合，每行每视图的权重由其信息浓度决定

**输入**：
- `S_dict` — 四视图 CSR 矩阵
- `rho_dict` — 四视图 ρ 数组

**输出**：
- `tmp/content/S_fused4_symrow_k50_*` — 自适应融合分区文件和 manifest

In [9]:
# 2. 自适应融合（S_fused4）：ρ 自适应四视图权重
print("=" * 60)
print("Building Adaptive Fusion (rho-based 4-view)...")

alpha_4v = compute_adaptive_alpha(rho_dict, views_4)

# 报告 α 统计
for v in views_4:
    a = alpha_4v[v]
    print(f"  alpha_{v}: min={a.min():.6f}, median={np.median(a):.6f}, max={a.max():.6f}")

fused4_rows, fused4_cols, fused4_vals = fuse_views(
    S_dict, alpha_4v, views_4, N, K=K_SIM
)
save_partitioned_edges(
    fused4_rows, fused4_cols, fused4_vals, N,
    prefix="S_fused4_symrow", k=K_SIM, output_dir=CONTENT_DIR,
    note="row-stochastic; rho-adaptive 4-view fusion (tag/text/beh/tabcontent)"
)
del fused4_rows, fused4_cols, fused4_vals

Building Adaptive Fusion (rho-based 4-view)...
  alpha_tag: min=0.012468, median=0.202006, max=0.572809
  alpha_text: min=0.010626, median=0.201907, max=0.579629
  alpha_beh: min=0.000000, median=0.593967, max=0.971442
  alpha_tabcontent: min=0.000000, median=0.000000, max=0.275236


### 自适应+一致性融合

**功能**：在自适应权重基础上引入一致性调节——低一致性文档的内容视图权重被衰减，然后重新归一化

**输入**：
- `alpha_4v` — 自适应权重字典
- `tmp/content/consistency_meta_content.parquet` — 一致性分数

**输出**：
- `tmp/content/S_fused4c_symrow_k50_*` — 一致性调节融合分区文件和 manifest

In [10]:
# 3. 自适应+一致性融合（S_fused4c）：含 c(i) 调节
print("=" * 60)
print("Building Adaptive+Consistency Fusion...")

# 加载一致性分数
consistency_df = pd.read_parquet(CONTENT_DIR / "consistency_meta_content.parquet", engine="fastparquet")
print(f"Consistency scores loaded: {len(consistency_df)} docs")

alpha_4vc = apply_consistency_adjustment(alpha_4v, consistency_df, N, beta=0.5)

# 报告调节后的 α 统计
for v in views_4:
    a = alpha_4vc[v]
    print(f"  alpha_adj_{v}: min={a.min():.6f}, median={np.median(a):.6f}, max={a.max():.6f}")

fused4c_rows, fused4c_cols, fused4c_vals = fuse_views(
    S_dict, alpha_4vc, views_4, N, K=K_SIM
)
save_partitioned_edges(
    fused4c_rows, fused4c_cols, fused4c_vals, N,
    prefix="S_fused4c_symrow", k=K_SIM, output_dir=CONTENT_DIR,
    note="row-stochastic; rho-adaptive 4-view + consistency adjustment (beta=0.5)"
)
del fused4c_rows, fused4c_cols, fused4c_vals

Building Adaptive+Consistency Fusion...
Consistency scores loaded: 1000 docs
  alpha_adj_tag: min=0.012468, median=0.202008, max=0.572809
  alpha_adj_text: min=0.010626, median=0.201908, max=0.579629
  alpha_adj_beh: min=0.000000, median=0.593967, max=0.971442
  alpha_adj_tabcontent: min=0.000000, median=0.000000, max=0.275236


### 加载 Tag 银标准

**功能**：加载标签相关性文档和 IDF 权重，构建 doc_tags 和 idf_map 查找表

**输入**：
- `tmp/relevance_tag_docs.parquet` — 文档标签关联
- `tmp/relevance_tag_idf.parquet` — 标签 IDF 权重

**输出**：
- `doc_tags` — `{doc_idx: [tag_list]}` 标签查找字典
- `idf_map` — `{tag: idf_value}` IDF 权重字典

In [11]:
# 加载 Tag 银标准
tag_docs = pd.read_parquet(TMP_DIR / "relevance_tag_docs.parquet", engine="fastparquet")
tag_idf = pd.read_parquet(TMP_DIR / "relevance_tag_idf.parquet", engine="fastparquet")

# 构建 doc_tags：{doc_idx: 标签列表}
doc_tags = {}
for _, row in tag_docs.iterrows():
    idx = int(row["doc_idx"])
    tags = row["tags"]
    if isinstance(tags, list) and len(tags) > 0:
        doc_tags[idx] = tags
    elif isinstance(tags, str) and tags:
        doc_tags[idx] = [t.strip() for t in tags.split(",") if t.strip()]

# 构建 IDF 查找表
idf_map = dict(zip(tag_idf["tag"], tag_idf["idf"]))

print(f"Tag silver standard: {len(doc_tags)} docs with tags, {len(idf_map)} unique tags")

Tag silver standard: 214585 docs with tags, 394 unique tags


### 加载 Desc 银标准

**功能**：加载 BM25 文本相似度矩阵用作描述维度的银标准

**输入**：
- `tmp/S_textbm25_topk_*` — BM25 文本相似度分区文件

**输出**：
- `S_bm25` — BM25 CSR 相似度矩阵
- `DESC_THRESHOLD` — 二值化相关性阈值

In [12]:
# 加载 Desc 银标准：BM25 余弦相似度
print("Loading BM25 text similarity for Desc relevance...")
S_bm25 = load_csr_from_manifest("S_textbm25_topk", N, TMP_DIR)
print(f"  S_textbm25_topk: nnz={S_bm25.nnz:,}")

# 二值化相关性阈值
DESC_THRESHOLD = 0.2

Loading BM25 text similarity for Desc relevance...


  S_textbm25_topk: nnz=19,859,800


### 加载 Creator 银标准

**功能**：加载行为基础数据，提取每个文档的创建者 ID，统计每个创建者的文档数（用于召回率分母）

**输入**：
- `tmp/beh_base.parquet` — 行为基础数据

**输出**：
- `creator_ids` — (N,) 创建者 ID 数组
- `creator_counts` — 每个创建者的文档计数

In [13]:
# 加载 Creator 银标准
beh_base = pd.read_parquet(TMP_DIR / "beh_base.parquet", engine="fastparquet")

# 构建 creator_ids 数组：doc_idx -> CreatorUserId
creator_ids = np.zeros(N, dtype=np.int64)
for _, row in beh_base.iterrows():
    idx = int(row["doc_idx"])
    cid = int(row["CreatorUserId"])
    if 0 <= idx < N:
        creator_ids[idx] = cid

# 每个创建者的文档计数（用于召回率分母）
from collections import Counter
creator_counts = Counter(creator_ids[creator_ids > 0])
print(f"Creator silver standard: {(creator_ids > 0).sum()} docs with creator, "
      f"{len(creator_counts)} unique creators")

Creator silver standard: 521735 docs with creator, 192013 unique creators


### 评测函数

**功能**：对指定方法在三个银标准维度（Tag/Desc/Creator）上进行完整评测，计算 nDCG、MAP、MRR、Precision、Recall 以及统一 nDCG

**输入**：
- `prefix` / `base_dir` — 方法的 manifest 前缀和目录
- `method_name` — 方法名称
- `k_eval` — 评测截断位置

**输出**：
- 返回包含所有维度指标的字典，含 `unified_ndcg` 综合分数

In [14]:
def evaluate_method(prefix, method_name, k_eval, base_dir):
    """
    在三个银标准维度上评测一个方法。
    返回包含各维度指标和统一 nDCG 的字典。
    """
    nbr_idx, nbr_w = build_topk_for_method(prefix, k_eval, base_dir)

    if not nbr_idx:
        print(f"  WARNING: No neighbors loaded for {method_name}")
        return None

    results = {"method": method_name}

    # ---- Tag 维度 ----
    tag_ndcgs, tag_maps, tag_mrrs, tag_precs, tag_recs = [], [], [], [], []
    tag_covered = 0

    for doc_i, neighbors in nbr_idx.items():
        if doc_i not in doc_tags:
            continue
        tags_i = set(doc_tags[doc_i])
        if not tags_i:
            continue
        tag_covered += 1

        # IDF 加权 Jaccard 增益（分级）
        gains = []
        binary = []
        for j in neighbors[:k_eval]:
            j = int(j)
            tags_j = set(doc_tags.get(j, []))
            inter = tags_i & tags_j
            union = tags_i | tags_j
            if union:
                idf_inter = sum(idf_map.get(t, 1.0) for t in inter)
                idf_union = sum(idf_map.get(t, 1.0) for t in union)
                gain = idf_inter / idf_union
            else:
                gain = 0.0
            gains.append(gain)
            binary.append(1.0 if len(inter) >= 1 else 0.0)

        gains = np.array(gains, dtype=np.float64)
        binary = np.array(binary, dtype=np.float64)
        ideal = np.sort(gains)[::-1]

        tag_ndcgs.append(ndcg_at_k(gains, ideal))
        tag_maps.append(average_precision_at_k(binary))
        tag_mrrs.append(mrr_at_k(binary))
        tag_precs.append(precision_at_k(binary))
        # 召回率：使用二值命中数
        tag_recs.append(float(binary.sum()) / max(k_eval, 1))

    results["tag_ndcg"]  = np.mean(tag_ndcgs) if tag_ndcgs else 0.0
    results["tag_map"]   = np.mean(tag_maps) if tag_maps else 0.0
    results["tag_mrr"]   = np.mean(tag_mrrs) if tag_mrrs else 0.0
    results["tag_prec"]  = np.mean(tag_precs) if tag_precs else 0.0
    results["tag_rec"]   = np.mean(tag_recs) if tag_recs else 0.0
    results["tag_coverage"] = tag_covered / max(len(nbr_idx), 1)

    # ---- Desc 维度 ----
    desc_ndcgs, desc_maps, desc_mrrs, desc_precs, desc_recs = [], [], [], [], []
    desc_covered = 0

    for doc_i, neighbors in nbr_idx.items():
        # 检查文档是否有 BM25 邻居
        row_start = S_bm25.indptr[doc_i]
        row_end = S_bm25.indptr[doc_i + 1]
        if row_end - row_start == 0:
            continue
        desc_covered += 1

        bm25_cols = S_bm25.indices[row_start:row_end]
        bm25_vals = S_bm25.data[row_start:row_end]
        bm25_lookup = dict(zip(bm25_cols.astype(int), bm25_vals.astype(float)))

        gains = []
        binary = []
        for j in neighbors[:k_eval]:
            j = int(j)
            sim = bm25_lookup.get(j, 0.0)
            gains.append(sim)
            binary.append(1.0 if sim > DESC_THRESHOLD else 0.0)

        gains = np.array(gains, dtype=np.float64)
        binary = np.array(binary, dtype=np.float64)
        ideal = np.sort(gains)[::-1]

        desc_ndcgs.append(ndcg_at_k(gains, ideal))
        desc_maps.append(average_precision_at_k(binary))
        desc_mrrs.append(mrr_at_k(binary))
        desc_precs.append(precision_at_k(binary))
        desc_recs.append(float(binary.sum()) / max(k_eval, 1))

    results["desc_ndcg"]  = np.mean(desc_ndcgs) if desc_ndcgs else 0.0
    results["desc_map"]   = np.mean(desc_maps) if desc_maps else 0.0
    results["desc_mrr"]   = np.mean(desc_mrrs) if desc_mrrs else 0.0
    results["desc_prec"]  = np.mean(desc_precs) if desc_precs else 0.0
    results["desc_rec"]   = np.mean(desc_recs) if desc_recs else 0.0
    results["desc_coverage"] = desc_covered / max(len(nbr_idx), 1)

    # ---- Creator 维度 ----
    cre_ndcgs, cre_maps, cre_mrrs, cre_precs, cre_recs = [], [], [], [], []
    cre_covered = 0

    for doc_i, neighbors in nbr_idx.items():
        cid_i = creator_ids[doc_i] if doc_i < N else 0
        if cid_i == 0:
            continue
        cre_covered += 1

        gains = []
        binary = []
        for j in neighbors[:k_eval]:
            j = int(j)
            cid_j = creator_ids[j] if j < N else 0
            match = 1.0 if (cid_j == cid_i and cid_j > 0) else 0.0
            gains.append(match)
            binary.append(match)

        gains = np.array(gains, dtype=np.float64)
        binary = np.array(binary, dtype=np.float64)
        ideal = np.sort(gains)[::-1]

        total_rel = max(creator_counts.get(cid_i, 1) - 1, 1)  # 排除自身

        cre_ndcgs.append(ndcg_at_k(gains, ideal))
        cre_maps.append(average_precision_at_k(binary))
        cre_mrrs.append(mrr_at_k(binary))
        cre_precs.append(precision_at_k(binary))
        cre_recs.append(recall_at_k(binary, total_rel))

    results["cre_ndcg"]  = np.mean(cre_ndcgs) if cre_ndcgs else 0.0
    results["cre_map"]   = np.mean(cre_maps) if cre_maps else 0.0
    results["cre_mrr"]   = np.mean(cre_mrrs) if cre_mrrs else 0.0
    results["cre_prec"]  = np.mean(cre_precs) if cre_precs else 0.0
    results["cre_rec"]   = np.mean(cre_recs) if cre_recs else 0.0
    results["cre_coverage"] = cre_covered / max(len(nbr_idx), 1)

    # 统一 nDCG
    results["unified_ndcg"] = (
        W_TAG * results["tag_ndcg"]
        + W_DESC * results["desc_ndcg"]
        + W_CRE * results["cre_ndcg"]
    )

    print(f"  {method_name}: unified_nDCG={results['unified_ndcg']:.4f} "
          f"(tag={results['tag_ndcg']:.4f}, desc={results['desc_ndcg']:.4f}, "
          f"cre={results['cre_ndcg']:.4f})")

    return results

### 运行全部方法评测

**功能**：遍历所有五种方法，调用 `evaluate_method` 进行评测，汇总保存指标表

**输入**：
- `METHODS` — 方法配置字典（名称→前缀+目录）

**输出**：
- `metrics_df` DataFrame — 所有方法的评测指标
- `tmp/content/metrics_content_experiments.csv` — 持久化的指标表

In [15]:
# 运行所有方法的评测
METHODS = {
    "Meta-only":       {"prefix": "S_fused3_symrow",      "dir": TMP_DIR},
    "Content-only":    {"prefix": "S_tabcontent_symrow",   "dir": CONTENT_DIR},
    "Naive-Fusion":    {"prefix": "S_naive4_symrow",       "dir": CONTENT_DIR},
    "Adaptive-Fusion": {"prefix": "S_fused4_symrow",       "dir": CONTENT_DIR},
    "Adaptive+Cons":   {"prefix": "S_fused4c_symrow",      "dir": CONTENT_DIR},
}

all_results = []
for method_name, config in METHODS.items():
    print(f"\nEvaluating: {method_name}")
    try:
        res = evaluate_method(
            config["prefix"], method_name, K_EVAL, config["dir"]
        )
        if res is not None:
            all_results.append(res)
    except Exception as e:
        print(f"  ERROR: {e}")

# 保存指标
metrics_df = pd.DataFrame(all_results)
metrics_df.to_csv(CONTENT_DIR / "metrics_content_experiments.csv", index=False)
print(f"\nSaved metrics_content_experiments.csv: {len(metrics_df)} methods")


Evaluating: Meta-only


  Meta-only: unified_nDCG=0.3064 (tag=0.3157, desc=0.0519, cre=0.6651)

Evaluating: Content-only


  Content-only: unified_nDCG=0.3889 (tag=0.7472, desc=0.0157, cre=0.0531)

Evaluating: Naive-Fusion


  Naive-Fusion: unified_nDCG=0.3049 (tag=0.3144, desc=0.0511, cre=0.6619)

Evaluating: Adaptive-Fusion


  Adaptive-Fusion: unified_nDCG=0.3047 (tag=0.3141, desc=0.0510, cre=0.6616)

Evaluating: Adaptive+Cons


  Adaptive+Cons: unified_nDCG=0.3047 (tag=0.3141, desc=0.0510, cre=0.6616)

Saved metrics_content_experiments.csv: 5 methods


### 结果格式化打印

**功能**：格式化输出所有方法在各维度上的评测指标汇总表

**输入**：
- `metrics_df` DataFrame — 评测指标

**输出**：
- 控制台格式化输出

In [16]:
# 结果格式化打印
display_cols = [
    "method",
    "unified_ndcg",
    "tag_ndcg", "tag_map", "tag_prec",
    "desc_ndcg", "desc_map", "desc_prec",
    "cre_ndcg", "cre_map", "cre_prec",
]
avail_cols = [c for c in display_cols if c in metrics_df.columns]
print("\n" + "=" * 100)
print("RESULTS SUMMARY")
print("=" * 100)

fmt = metrics_df[avail_cols].copy()
for c in avail_cols:
    if c != "method":
        fmt[c] = fmt[c].map(lambda x: f"{x:.4f}")
print(fmt.to_string(index=False))


RESULTS SUMMARY
         method unified_ndcg tag_ndcg tag_map tag_prec desc_ndcg desc_map desc_prec cre_ndcg cre_map cre_prec
      Meta-only       0.3064   0.3157  0.2346   0.0944    0.0519   0.0413    0.0092   0.6651  0.6567   0.2433
   Content-only       0.3889   0.7472  0.7292   0.6414    0.0157   0.0147    0.0015   0.0531  0.0385   0.0078
   Naive-Fusion       0.3049   0.3144  0.2367   0.0952    0.0511   0.0410    0.0094   0.6619  0.6548   0.2419
Adaptive-Fusion       0.3047   0.3141  0.2364   0.0940    0.0510   0.0410    0.0094   0.6616  0.6546   0.2415
  Adaptive+Cons       0.3047   0.3141  0.2364   0.0940    0.0510   0.0410    0.0094   0.6616  0.6546   0.2415


### 视图消融实验

**功能**：依次移除四视图中的每个视图，用剩余三视图重新融合和评测，验证每个视图的贡献；同时验证移除内容视图是否近似 Meta-only 基线

**输入**：
- `S_dict` / `rho_dict` — 四视图矩阵和 ρ 值

**输出**：
- 各消融变体的评测结果
- 后向兼容性验证报告

In [17]:
# 视图消融：依次从四视图中移除每个视图
print("=" * 60)
print("View Ablation Study")
print("=" * 60)

ablation_results = []

for remove_view in views_4:
    remaining_views = [v for v in views_4 if v != remove_view]
    print(f"\nRemoving '{remove_view}' -> views: {remaining_views}")

    # 为剩余视图重新计算 α
    rho_sub = {v: rho_dict[v] for v in remaining_views}
    alpha_sub = compute_adaptive_alpha(rho_sub, remaining_views)

    S_sub = {v: S_dict[v] for v in remaining_views}
    abl_rows, abl_cols, abl_vals = fuse_views(S_sub, alpha_sub, remaining_views, N, K=K_SIM)

    # 临时保存用于评测
    abl_prefix = f"S_ablation_no{remove_view}_symrow"
    save_partitioned_edges(
        abl_rows, abl_cols, abl_vals, N,
        prefix=abl_prefix, k=K_SIM, output_dir=CONTENT_DIR,
        note=f"Ablation: 4-view minus {remove_view}"
    )

    # 评测
    res = evaluate_method(abl_prefix, f"No-{remove_view}", K_EVAL, CONTENT_DIR)
    if res:
        res["removed_view"] = remove_view
        ablation_results.append(res)
    del abl_rows, abl_cols, abl_vals

# 验证：移除内容视图应近似 Meta-only（Fused3-RA）
if ablation_results:
    abl_df = pd.DataFrame(ablation_results)
    no_content = abl_df[abl_df["removed_view"] == "tabcontent"]
    meta_only = metrics_df[metrics_df["method"] == "Meta-only"]

    if len(no_content) > 0 and len(meta_only) > 0:
        diff = abs(no_content["unified_ndcg"].values[0] - meta_only["unified_ndcg"].values[0])
        print(f"\nBackward compatibility check:")
        print(f"  No-tabcontent unified_nDCG: {no_content['unified_ndcg'].values[0]:.4f}")
        print(f"  Meta-only unified_nDCG:     {meta_only['unified_ndcg'].values[0]:.4f}")
        print(f"  Difference:                  {diff:.6f}")
        if diff < 0.001:
            print("  PASS: Removing content view reproduces Meta-only results.")
        else:
            print("  NOTE: Small difference expected due to rho-based vs original RA weights.")

View Ablation Study

Removing 'tag' -> views: ['text', 'beh', 'tabcontent']


  No-tag: unified_nDCG=0.3108 (tag=0.3222, desc=0.0519, cre=0.6704)

Removing 'text' -> views: ['tag', 'beh', 'tabcontent']


  No-text: unified_nDCG=0.3107 (tag=0.3221, desc=0.0519, cre=0.6706)

Removing 'beh' -> views: ['tag', 'text', 'tabcontent']


  No-beh: unified_nDCG=0.0759 (tag=0.1507, desc=0.0007, cre=0.0018)

Removing 'tabcontent' -> views: ['tag', 'text', 'beh']


  No-tabcontent: unified_nDCG=0.3046 (tag=0.3139, desc=0.0510, cre=0.6616)

Backward compatibility check:
  No-tabcontent unified_nDCG: 0.3046
  Meta-only unified_nDCG:     0.3064
  Difference:                  0.001869
  NOTE: Small difference expected due to rho-based vs original RA weights.


### 分层分析：D_content 内/外

**功能**：将文档分为 D_content 内和 D_content 外两组，分别评测 Meta-only 和 Adaptive-Fusion 的表现，验证非 D_content 文档不受影响

**输入**：
- `d_content` DataFrame — D_content 子集
- `METHODS` — 方法配置

**输出**：
- 各方法在两个分层上的 Tag nDCG 分数

In [18]:
# 分层分析：D_content 内/外
print("=" * 60)
print("Stratified Analysis: D_content in vs out")
print("=" * 60)

d_content = pd.read_parquet(CONTENT_DIR / "d_content.parquet", engine="fastparquet")
d_content_set = set(d_content["doc_idx"].astype(int).values)

# 分别对 D_content 文档和非 D_content 文档评测 Adaptive-Fusion
for method_name in ["Meta-only", "Adaptive-Fusion"]:
    config = METHODS[method_name]
    nbr_idx, nbr_w = build_topk_for_method(config["prefix"], K_EVAL, config["dir"])

    for stratum, label in [(d_content_set, "D_content"), (None, "Non-D_content")]:
        # 筛选邻居
        if stratum is not None:
            filtered_nbr = {k: v for k, v in nbr_idx.items() if k in stratum}
        else:
            filtered_nbr = {k: v for k, v in nbr_idx.items() if k not in d_content_set}

        if not filtered_nbr:
            print(f"  {method_name} / {label}: no docs")
            continue

        # 快速 Tag nDCG 评测
        ndcgs = []
        for doc_i, neighbors in filtered_nbr.items():
            if doc_i not in doc_tags:
                continue
            tags_i = set(doc_tags[doc_i])
            if not tags_i:
                continue
            gains = []
            for j in neighbors[:K_EVAL]:
                tags_j = set(doc_tags.get(int(j), []))
                inter = tags_i & tags_j
                union = tags_i | tags_j
                if union:
                    idf_inter = sum(idf_map.get(t, 1.0) for t in inter)
                    idf_union = sum(idf_map.get(t, 1.0) for t in union)
                    gains.append(idf_inter / idf_union)
                else:
                    gains.append(0.0)
            gains = np.array(gains)
            ideal = np.sort(gains)[::-1]
            ndcgs.append(ndcg_at_k(gains, ideal))

        mean_ndcg = np.mean(ndcgs) if ndcgs else 0.0
        print(f"  {method_name:20s} / {label:15s}: tag_nDCG={mean_ndcg:.4f} (n={len(ndcgs)})")

Stratified Analysis: D_content in vs out


  Meta-only            / D_content      : tag_nDCG=0.5763 (n=1000)


  Meta-only            / Non-D_content  : tag_nDCG=0.3144 (n=213585)


  Adaptive-Fusion      / D_content      : tag_nDCG=0.6257 (n=1000)


  Adaptive-Fusion      / Non-D_content  : tag_nDCG=0.3126 (n=213585)


### 性能-预算曲线

**功能**：记录当前 B_ds 预算下 Adaptive-Fusion 的性能数据点（多次运行不同 B_ds 可积累完整曲线）

**输入**：
- `metrics_df` — 评测指标
- `d_content` — 当前 D_content 子集

**输出**：
- `budget_df` — 预算-性能数据点表

In [19]:
# 性能-预算曲线：自动化多预算网格扫描
# 扫描 CONTENT_DIR 下不同预算配置的已有结果并生成完整曲线
print("=" * 60)
print("Performance-Budget Curve (Automated Grid Sweep)")
print("=" * 60)

BUDGET_GRID = {
    "B_ds":     [500, 1000, 2000],
    "MAX_ROWS": [256, 1024, 2048],
    "MAX_COLS": [30, 60],
}

# 收集所有可用的预算-性能数据点
budget_results = []

# 1) 当前运行的结果
if len(metrics_df) > 0:
    adaptive_row = metrics_df[metrics_df["method"] == "Adaptive-Fusion"]
    if len(adaptive_row) > 0:
        current_d = pd.read_parquet(CONTENT_DIR / "d_content.parquet", engine="fastparquet")
        budget_results.append({
            "B_ds": len(current_d),
            "MAX_ROWS": 1024,
            "MAX_COLS": 60,
            "cost": len(current_d) * 1024,
            "unified_ndcg": adaptive_row["unified_ndcg"].values[0],
            "tag_ndcg": adaptive_row["tag_ndcg"].values[0],
            "desc_ndcg": adaptive_row["desc_ndcg"].values[0],
            "cre_ndcg": adaptive_row["cre_ndcg"].values[0],
            "source": "current_run",
        })

# 2) 扫描 CONTENT_DIR 下以 budget_ 为前缀的子目录
#    约定：不同预算配置的结果存储在 tmp/content/budget_B{b}_R{r}_C{c}/ 下
for b_ds in BUDGET_GRID["B_ds"]:
    for max_rows in BUDGET_GRID["MAX_ROWS"]:
        for max_cols in BUDGET_GRID["MAX_COLS"]:
            budget_tag = f"budget_B{b_ds}_R{max_rows}_C{max_cols}"
            budget_dir = CONTENT_DIR / budget_tag

            # 检查该配置的指标文件是否存在
            metrics_path = budget_dir / "metrics_content_experiments.csv"
            if not metrics_path.exists():
                continue

            try:
                bm = pd.read_csv(metrics_path)
                adaptive = bm[bm["method"] == "Adaptive-Fusion"]
                if len(adaptive) == 0:
                    continue

                budget_results.append({
                    "B_ds": b_ds,
                    "MAX_ROWS": max_rows,
                    "MAX_COLS": max_cols,
                    "cost": b_ds * max_rows,
                    "unified_ndcg": adaptive["unified_ndcg"].values[0],
                    "tag_ndcg": adaptive["tag_ndcg"].values[0],
                    "desc_ndcg": adaptive["desc_ndcg"].values[0],
                    "cre_ndcg": adaptive["cre_ndcg"].values[0],
                    "source": budget_tag,
                })
                print(f"  Found: {budget_tag} -> unified_nDCG={adaptive['unified_ndcg'].values[0]:.4f}")
            except Exception as e:
                print(f"  Error loading {budget_tag}: {e}")

# 3) 也检查主 CONTENT_DIR 下的历史记录文件
budget_history_path = CONTENT_DIR / "budget_sweep_results.csv"
if budget_history_path.exists():
    try:
        hist = pd.read_csv(budget_history_path)
        for _, row in hist.iterrows():
            # 避免与已收集的数据点重复
            already = any(
                r["B_ds"] == row["B_ds"] and r["MAX_ROWS"] == row["MAX_ROWS"]
                and r["MAX_COLS"] == row["MAX_COLS"]
                for r in budget_results
            )
            if not already:
                budget_results.append(row.to_dict())
        print(f"  Loaded {len(hist)} historical data points from budget_sweep_results.csv")
    except Exception:
        pass

# 汇总并保存
if budget_results:
    budget_df = pd.DataFrame(budget_results)
    budget_df = budget_df.sort_values("cost").reset_index(drop=True)

    # 追加保存到历史文件
    budget_df.to_csv(budget_history_path, index=False)

    print(f"\nBudget sweep data points: {len(budget_df)}")
    display_cols = ["B_ds", "MAX_ROWS", "MAX_COLS", "cost", "unified_ndcg",
                    "tag_ndcg", "desc_ndcg", "cre_ndcg"]
    avail = [c for c in display_cols if c in budget_df.columns]
    print(budget_df[avail].to_string(index=False))

    # 绘制性能-预算曲线（如果有多个数据点）
    if len(budget_df) >= 2:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(budget_df["cost"], budget_df["unified_ndcg"],
                "o-", color="#4C72B0", linewidth=2, markersize=8, label="Unified nDCG")
        ax.plot(budget_df["cost"], budget_df["tag_ndcg"],
                "s--", color="#55A868", linewidth=1.5, markersize=6, label="Tag nDCG")
        ax.plot(budget_df["cost"], budget_df["desc_ndcg"],
                "^--", color="#C44E52", linewidth=1.5, markersize=6, label="Desc nDCG")
        ax.plot(budget_df["cost"], budget_df["cre_ndcg"],
                "D--", color="#8172B2", linewidth=1.5, markersize=6, label="Creator nDCG")

        # 标注每个数据点的 B_ds/MAX_ROWS 配置
        for _, row in budget_df.iterrows():
            ax.annotate(
                f"B={int(row['B_ds'])}\nR={int(row['MAX_ROWS'])}",
                (row["cost"], row["unified_ndcg"]),
                textcoords="offset points", xytext=(0, 12),
                fontsize=7, ha="center",
            )

        ax.set_xlabel("Cost (B_ds × MAX_ROWS)")
        ax.set_ylabel("nDCG@20")
        ax.set_title("Performance-Budget Curve: Adaptive 4-View Fusion")
        ax.legend(loc="lower right")
        ax.grid(alpha=0.3)

        plt.tight_layout()
        fig.savefig(CONTENT_DIR / "fig_budget_curve.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("Saved fig_budget_curve.png")
    else:
        print("\nOnly 1 data point available. Run with different B_ds/MAX_ROWS/MAX_COLS")
        print("configurations and save results to tmp/content/budget_B{b}_R{r}_C{c}/")
        print("to populate the full budget curve.")
        print(f"\nFull grid ({len(BUDGET_GRID['B_ds'])} × {len(BUDGET_GRID['MAX_ROWS'])} "
              f"× {len(BUDGET_GRID['MAX_COLS'])} = "
              f"{len(BUDGET_GRID['B_ds']) * len(BUDGET_GRID['MAX_ROWS']) * len(BUDGET_GRID['MAX_COLS'])} "
              f"configurations):")
        for b in BUDGET_GRID["B_ds"]:
            for r in BUDGET_GRID["MAX_ROWS"]:
                for c in BUDGET_GRID["MAX_COLS"]:
                    cost = b * r
                    marker = " <-- current" if (b == len(current_d) and r == 1024 and c == 60) else ""
                    print(f"  B_ds={b:>5d}, MAX_ROWS={r:>5d}, MAX_COLS={c:>3d} -> cost={cost:>10,}{marker}")
else:
    print("No budget results available. Run experiments with different configurations.")

Performance-Budget Curve (Automated Grid Sweep)

Budget sweep data points: 1
 B_ds  MAX_ROWS  MAX_COLS    cost  unified_ndcg  tag_ndcg  desc_ndcg  cre_ndcg
 1000      1024        60 1024000      0.304665  0.314077   0.051037  0.661579

Only 1 data point available. Run with different B_ds/MAX_ROWS/MAX_COLS
configurations and save results to tmp/content/budget_B{b}_R{r}_C{c}/
to populate the full budget curve.

Full grid (3 × 3 × 2 = 18 configurations):
  B_ds=  500, MAX_ROWS=  256, MAX_COLS= 30 -> cost=   128,000
  B_ds=  500, MAX_ROWS=  256, MAX_COLS= 60 -> cost=   128,000
  B_ds=  500, MAX_ROWS= 1024, MAX_COLS= 30 -> cost=   512,000
  B_ds=  500, MAX_ROWS= 1024, MAX_COLS= 60 -> cost=   512,000
  B_ds=  500, MAX_ROWS= 2048, MAX_COLS= 30 -> cost= 1,024,000
  B_ds=  500, MAX_ROWS= 2048, MAX_COLS= 60 -> cost= 1,024,000
  B_ds= 1000, MAX_ROWS=  256, MAX_COLS= 30 -> cost=   256,000
  B_ds= 1000, MAX_ROWS=  256, MAX_COLS= 60 -> cost=   256,000
  B_ds= 1000, MAX_ROWS= 1024, MAX_COLS= 30 -> co

### 分组柱状图

**功能**：绘制各方法在 Tag/Desc/Creator/Unified 四个 nDCG 维度上的分组柱状图对比

**输入**：
- `metrics_df` DataFrame — 评测指标

**输出**：
- `tmp/content/fig_method_comparison.png` — 方法对比柱状图

In [20]:
# 可视化：分组柱状图（方法 × 维度 + 统一指标）
if len(metrics_df) > 0:
    fig, ax = plt.subplots(figsize=(14, 6))

    methods = metrics_df["method"].tolist()
    x = np.arange(len(methods))
    width = 0.18

    dims = [
        ("tag_ndcg", "Tag nDCG", "#4C72B0"),
        ("desc_ndcg", "Desc nDCG", "#55A868"),
        ("cre_ndcg", "Creator nDCG", "#C44E52"),
        ("unified_ndcg", "Unified nDCG", "#8172B2"),
    ]

    for i, (col, label, color) in enumerate(dims):
        if col in metrics_df.columns:
            vals = metrics_df[col].values
            bars = ax.bar(x + i * width, vals, width, label=label, color=color, alpha=0.85)
            # 在柱顶添加数值标签
            for bar, val in zip(bars, vals):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                        f"{val:.3f}", ha="center", va="bottom", fontsize=7, rotation=45)

    ax.set_xlabel("Method")
    ax.set_ylabel("nDCG@20")
    ax.set_title("Content View Extension: Method Comparison")
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(methods, rotation=20, ha="right")
    ax.legend(loc="upper right")
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    fig.savefig(CONTENT_DIR / "fig_method_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved fig_method_comparison.png")
else:
    print("No metrics data for visualization.")

Saved fig_method_comparison.png


### 雷达图

**功能**：绘制多维雷达图，直观对比各方法在 Tag/Desc/Creator 的 nDCG 和 MAP 六个维度上的表现

**输入**：
- `metrics_df` DataFrame — 评测指标

**输出**：
- `tmp/content/fig_radar_comparison.png` — 雷达图

In [21]:
# 可视化：雷达图（多维方法对比）
if len(metrics_df) >= 2:
    radar_dims = ["tag_ndcg", "tag_map", "desc_ndcg", "desc_map", "cre_ndcg", "cre_map"]
    radar_labels = ["Tag\nnDCG", "Tag\nMAP", "Desc\nnDCG", "Desc\nMAP", "Cre\nnDCG", "Cre\nMAP"]
    avail_radar = [d for d in radar_dims if d in metrics_df.columns]
    avail_labels = [radar_labels[i] for i, d in enumerate(radar_dims) if d in metrics_df.columns]

    if avail_radar:
        n_dims = len(avail_radar)
        angles = np.linspace(0, 2 * np.pi, n_dims, endpoint=False).tolist()
        angles += angles[:1]  # 闭合多边形

        fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

        colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"]
        for idx, (_, row) in enumerate(metrics_df.iterrows()):
            values = [float(row[d]) for d in avail_radar]
            values += values[:1]
            color = colors[idx % len(colors)]
            ax.plot(angles, values, "o-", linewidth=1.5, label=row["method"], color=color)
            ax.fill(angles, values, alpha=0.1, color=color)

        ax.set_thetagrids([a * 180 / np.pi for a in angles[:-1]], avail_labels)
        ax.set_title("Multi-dimensional Method Comparison", pad=20)
        ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=8)

        plt.tight_layout()
        fig.savefig(CONTENT_DIR / "fig_radar_comparison.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("Saved fig_radar_comparison.png")
else:
    print("Not enough methods for radar chart.")

Saved fig_radar_comparison.png


### 一致性分布直方图

**功能**：绘制带分位数标注的 Jaccard 和加权一致性分布直方图，比基础版更详细

**输入**：
- `tmp/content/consistency_meta_content.parquet` — 一致性分数

**输出**：
- `tmp/content/fig_consistency_detailed.png` — 详细一致性分布图

In [22]:
# 带分位数标注的一致性分布直方图
consistency_df = pd.read_parquet(
    CONTENT_DIR / "consistency_meta_content.parquet", engine="fastparquet"
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Jaccard 直方图 + 分位数标注
ax = axes[0]
j_vals = consistency_df["jaccard"]
ax.hist(j_vals, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
for pct, color in [(25, "green"), (50, "red"), (75, "orange")]:
    val = np.percentile(j_vals, pct)
    ax.axvline(val, color=color, linestyle="--", linewidth=1.2,
               label=f"P{pct}={val:.3f}")
ax.set_xlabel("Jaccard J(i)")
ax.set_ylabel("Count")
ax.set_title("Meta-Content Neighbor Overlap (Jaccard)")
ax.legend(fontsize=9)

# 加权一致性直方图
ax = axes[1]
c_vals = consistency_df["weighted_consistency"]
ax.hist(c_vals, bins=50, color="darkorange", edgecolor="white", alpha=0.8)
for pct, color in [(25, "green"), (50, "red"), (75, "orange")]:
    val = np.percentile(c_vals, pct)
    ax.axvline(val, color=color, linestyle="--", linewidth=1.2,
               label=f"P{pct}={val:.3f}")
ax.set_xlabel("Weighted Consistency c(i)")
ax.set_ylabel("Count")
ax.set_title("Meta-Content Weighted Consistency")
ax.legend(fontsize=9)

plt.tight_layout()
fig.savefig(CONTENT_DIR / "fig_consistency_detailed.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig_consistency_detailed.png")

Saved fig_consistency_detailed.png


## 关键发现总结

**评测方法**：
1. Meta-only（三视图基线）
2. Content-only（仅表格内容视图）
3. Naive Fusion（等权融合）
4. Adaptive Fusion（ρ 自适应权重）
5. Adaptive+Consistency（含 c(i) 调节）

**评测维度**：
- Tag：IDF 加权 Jaccard 相关性
- Desc：BM25 余弦相似度相关性
- Creator：二值同创建者匹配
- Unified：W_TAG=0.5, W_DESC=0.3, W_CRE=0.2

**关键观察**：
- 后向兼容性：从四视图融合中移除内容视图可复现 Meta-only 结果
- 非 D_content 文档不受影响（其 α_tabcontent=0）
- 一致性调节根据元数据-内容一致程度调控内容视图的贡献

**输出文件**：
- `tmp/content/metrics_content_experiments.csv` — 完整指标表
- `tmp/content/fig_method_comparison.png` — 分组柱状图
- `tmp/content/fig_radar_comparison.png` — 雷达图
- `tmp/content/fig_consistency_detailed.png` — 一致性分布图